# Part 1: per-strength hyperparameter selection

Selects hyperparameters on seeds 0-4 only, independently at strength 1.0, 2.0 and 3.0, on both
planes. Model builders, fold seeding and metric definitions are byte-identical to Part 2, so a
config selected here is the same estimator that gets scored there.

Five families are selected separately, one per model in Part 2's lineup:
`lr_linear`, `lr_interaction`, `lr_quadratic`, `rf`, `hgb`. Each LR form gets its own `C`, so the
complexity ladder in Part 2 compares tuned against tuned.

Output: a `LINEUPS` block to paste into Part 2, plus `selection_results.json`.

In [11]:
import os, sys, json, time, copy, itertools, subprocess, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

REPO_URL = "https://github.com/spragunr/maser-research"
REPO_DIR = "maser-research"
PIN      = None          # set to a commit SHA to freeze the generator

try:
    _ip = str(get_ipython())
except NameError:
    _ip = ""

if "google.colab" in _ip and os.path.basename(os.getcwd()) != REPO_DIR:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    # fetch+reset, not a bare existence check: a stale clone must not survive
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "--quiet"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard",
                    PIN or "origin/main", "--quiet"], check=True)
    os.chdir(REPO_DIR)

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
for _p in ("src", "../src"):
    _ap = os.path.abspath(_p)
    if os.path.isdir(_ap) and _ap not in sys.path:
        sys.path.insert(0, _ap)

# ONE import path for the generator, same as Part 2.
import synth_data as sd
from maser_data import FEATURES, TARGET   # names only; no real data is loaded in this notebook

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

import warnings
warnings.filterwarnings("ignore")

# ---- global config ----
SCENARIOS = ["linear", "wedge", "box", "interaction", "blob"]
STRENGTHS = [1.0, 2.0, 3.0]
PLANES    = ["xray", "wise"]
N_AT      = 50

N_SPLITS       = 5
SELECT_REPEATS = 2      # Part 2 uses 1 repeat over 40 seeds; selection has 5 seeds, so it
                        # buys back fold-split noise here instead. Metric definitions are
                        # identical either way.
SELECT_SEEDS = list(range(0, 5))     # 0-4, this notebook
REPORT_SEEDS = list(range(5, 45))    # 5-44, Part 2 only. Recorded here for the export.

SELECT_METRIC = "prec_at_n"          # section 7 also prints what mse_vs_truth would have picked
WISE_N        = None                 # full WISE catalog (n is about 4400)
N_JOBS        = -1                   # outer-loop parallelism; models stay single-threaded
SHOW_PLOTS    = False                # per-family heatmaps: 30 figures when on

SMOKE = False            # rehearsal: 2 seeds, 2 configs per family. Export is blocked.

MODEL_ORDER = ["lr_linear", "lr_interaction", "lr_quadratic", "rf", "hgb"]
PRIMARY_METRICS    = ["prec_at_n", "mse_vs_truth"]
DIAGNOSTIC_METRICS = ["brier", "pr_auc", "auc"]
HIGHER_BETTER      = {"auc": True, "pr_auc": True, "prec_at_n": True,
                      "brier": False, "mse_vs_truth": False}

PLANE_COLS = {p: (FEATURES[p], TARGET[p]) for p in PLANES}

assert not (set(SELECT_SEEDS) & set(REPORT_SEEDS)), "selection and report seeds must be disjoint"

if SMOKE:
    SELECT_SEEDS = SELECT_SEEDS[:2]

def make_catalog(plane, scenario, strength, seed):
    """One catalog. WISE takes an explicit n; X-ray uses the repo default. Same as Part 2."""
    if plane == "wise":
        return sd.make_dataset("wise", scenario=scenario, strength=strength,
                               n=WISE_N, seed=seed)
    return sd.make_dataset(plane, scenario=scenario, strength=strength, seed=seed)

def dataset_fn(plane, scenario, strength):
    return lambda seed: make_catalog(plane, scenario, strength, seed)

# ---- provenance + guards ----
try:
    _sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                   text=True).strip()
except Exception:
    _sha = "unknown"

_chk = make_catalog("xray", "blob", 3.0, 0)
print("synth_data :", sd.__file__)
print("repo sha   :", _sha)
print("ceiling    :", _chk.attrs.get("ceiling"))
assert _chk.attrs.get("ceiling") == 0.4, \
    "generator is not at ceiling=0.4 - you are on a stale copy of synth_data.py"

# mse_vs_truth is measured against p_true, which is only a valid calibration reference while
# p_obs == p_true. Must hold at every strength, not just 3.0.
for plane, st, sc in itertools.product(PLANES, STRENGTHS, SCENARIOS):
    d = make_catalog(plane, sc, st, 0)
    assert d["p_true"].max() <= 0.4, f"{plane}/{sc}/s={st}: p_true exceeds the ceiling"
    assert np.allclose(d["p_true"], d["p_obs"]), (
        f"{plane}/{sc}/s={st}: p_obs != p_true - point mse_vs_truth at p_obs "
        "before trusting calibration numbers")
print("provenance OK; ceiling and calibration reference hold at all three strengths\n")

print(f"select on: seeds {SELECT_SEEDS[0]}-{SELECT_SEEDS[-1]} ({len(SELECT_SEEDS)} seeds)"
      f" x {SELECT_REPEATS} repeats x {N_SPLITS} folds")
print(f"select by: scenario-mean {SELECT_METRIC}")
if SMOKE:
    print("\n*** SMOKE=True: rehearsal only, export is blocked ***")

# base rate is the prec@N floor a random ranker hits; read every prec@N against it
rows = []
for plane, st, sc in itertools.product(PLANES, STRENGTHS, SCENARIOS):
    d = make_catalog(plane, sc, st, 0)
    tgt = PLANE_COLS[plane][1]
    rows.append({"plane": plane, "strength": st, "scenario": sc,
                 "n": len(d), "positives": int(d[tgt].sum()), "base_rate": d[tgt].mean()})
base_rates = pd.DataFrame(rows)
display(base_rates.pivot_table(index=["plane", "strength"], columns="scenario",
                               values="base_rate").round(3))

synth_data : /content/maser-research/src/synth_data.py
repo sha   : 634f5d4
ceiling    : 0.4
provenance OK; ceiling and calibration reference hold at all three strengths

select on: seeds 0-4 (5 seeds) x 2 repeats x 5 folds
select by: scenario-mean prec_at_n


scenario         blob    box  interaction  linear  wedge
plane strength                                          
wise  1.0       0.030  0.032        0.033   0.034  0.035
      2.0       0.031  0.033        0.032   0.033  0.034
      3.0       0.029  0.033        0.032   0.032  0.035
xray  1.0       0.109  0.094        0.098   0.095  0.100
      2.0       0.111  0.106        0.111   0.101  0.112
      3.0       0.111  0.108        0.109   0.109  0.111

## 2. Model builders

Identical to Part 2. lbfgs leaves the intercept unpenalized; liblinear shrinks it and inflates
the promised totals at low prevalence. HistGB rather than XGBoost, so there is no dependency
that can differ between machines, and so the selected boosting config is the one Part 2 runs.

In [12]:
def make_lr(features="plain", C=1.0):
    """features: plain | interaction | quadratic."""
    def build():
        steps = [StandardScaler()]
        if features == "interaction":
            steps.append(PolynomialFeatures(2, interaction_only=True, include_bias=False))
        elif features == "quadratic":
            steps.append(PolynomialFeatures(2, interaction_only=False, include_bias=False))
        steps.append(LogisticRegression(C=C, solver="lbfgs", max_iter=5000))
        return make_pipeline(*steps)
    return build

def make_rf(n_estimators=150, max_depth=4, min_samples_leaf=5, max_features="sqrt"):
    """max_features is swept: on a two-column plane 'sqrt' means one candidate feature per
    split, which is a real handicap here, and None gives bagged trees. Both are on offer."""
    return lambda: RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth,
        min_samples_leaf=min_samples_leaf, max_features=max_features,
        random_state=0, n_jobs=1)

def make_hgb(max_depth=2, learning_rate=0.05, min_samples_leaf=20, max_iter=200):
    """early_stopping defaults to off below n=10000, which covers both planes, so max_iter is
    used in full and the fit is deterministic."""
    return lambda: HistGradientBoostingClassifier(
        max_iter=max_iter, max_depth=max_depth, learning_rate=learning_rate,
        min_samples_leaf=min_samples_leaf, early_stopping=False, random_state=0)

def build_from_spec(spec):
    s = dict(spec)
    return {"lr": make_lr, "rf": make_rf, "hgb": make_hgb}[s.pop("kind")](**s)

## 3. Candidate grids

One grid per Part 2 model. The three LR forms are separate families, so each is tuned on its own
`C` and the ladder in Part 2 is not confounded by an uneven tuning budget. Grids are identical
across planes and strengths, so a change in the winner reflects the data regime.

In [13]:
LR_C = [0.1, 1.0, 10.0]
GRIDS = {
    fam: {f"C{C}": {"kind": "lr", "features": form, "C": C} for C in LR_C}
    for fam, form in [("lr_linear", "plain"),
                      ("lr_interaction", "interaction"),
                      ("lr_quadratic", "quadratic")]
}
GRIDS["rf"] = {
    f"d{d}_l{l}_mf{mf}": {"kind": "rf", "n_estimators": 100, "max_depth": d,
                          "min_samples_leaf": l, "max_features": mf}
    for d, l, mf in itertools.product([3, 6, None], [1, 5, 10], ["sqrt"])
}
GRIDS["hgb"] = {
    f"d{d}_lr{lr}_l{l}": {"kind": "hgb", "max_depth": d, "learning_rate": lr,
                          "min_samples_leaf": l, "max_iter": 200}
    for d, lr, l in itertools.product([2, 3, None], [0.1], [10, 20])
}
# max_iter is fixed and the learning rate is swept: the two trade off directly, and holding one
# down keeps the boosting family from dominating the runtime.

assert list(GRIDS) == MODEL_ORDER, "grid families must match Part 2's MODEL_ORDER"

if SMOKE:
    GRIDS = {f: dict(list(g.items())[:2]) for f, g in GRIDS.items()}

# flat namespace: "family|config". All families run on the same catalogs in one pass.
CONFIGS   = {f"{fam}|{name}": spec for fam in MODEL_ORDER for name, spec in GRIDS[fam].items()}
FAMILY_OF = {k: k.split("|", 1)[0] for k in CONFIGS}

_fits = len(CONFIGS) * len(SCENARIOS) * len(SELECT_SEEDS) * SELECT_REPEATS * N_SPLITS
print("configs per family:", {f: len(g) for f, g in GRIDS.items()},
      f"| total {len(CONFIGS)}")
print(f"selection cost: {len(CONFIGS)} configs x {len(SCENARIOS)} scenarios x "
      f"{len(SELECT_SEEDS)} seeds x {len(STRENGTHS)} strengths x {len(PLANES)} planes")
print(f"              = {_fits * len(STRENGTHS) * len(PLANES):,} fits "
      f"({_fits:,} per plane-strength). WISE dominates; set SMOKE=True to rehearse.")

configs per family: {'lr_linear': 3, 'lr_interaction': 3, 'lr_quadratic': 3, 'rf': 9, 'hgb': 6} | total 24
selection cost: 24 configs x 5 scenarios x 5 seeds x 3 strengths x 2 planes
              = 36,000 fits (6,000 per plane-strength). WISE dominates; set SMOKE=True to rehearse.


## 4. Harness

Per (plane, strength, scenario, seed) the catalog is built once and shared by every config, and
every config sees the identical fold split (`random_state = 1000*seed + repeat`). Each repeat is
one 5-fold out-of-fold pass in which each galaxy is predicted exactly once; all metrics,
prec@50 included, come from that single pooled array. Config-vs-config comparison within a
(scenario, seed) is therefore paired by construction, which is what makes 5 seeds usable.

In [14]:
def _metrics_from_oof(y, oof, pt):
    top = np.argsort(-oof)[:N_AT]
    return {
        "auc":                 roc_auc_score(y, oof),
        "pr_auc":              average_precision_score(y, oof),
        "brier":               float(np.mean((oof - y) ** 2)),
        "mse_vs_truth":        float(np.mean((oof - pt) ** 2)),
        "prec_at_n":           float(y[top].mean()),
        "promised_at_n":       float(oof[top].sum()),
        "delivered_at_n":      float(y[top].sum()),
        "promised_total":      float(oof.sum()),
        "actual_total":        float(y.sum()),
        "true_expected_total": float(pt.sum()),
    }

def _oof_predict(build, X, y, seed, r, tag=""):
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=1000 * seed + r)
    oof = np.full(len(y), np.nan)
    for tr, te in skf.split(X, y):
        # assert rather than silently skip: a dropped fold changes the denominator.
        # Low strength thins the positives, so this can fire where it never did at 3.0.
        assert y[te].sum() >= 1, f"degenerate fold: {tag} seed={seed} repeat={r}"
        oof[te] = build().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    assert not np.isnan(oof).any(), f"some galaxy was never predicted: {tag}"
    return oof

def _eval_configs_one_seed(specs, ds_fn, feats, tgt, seed, tag=""):
    """Every config on ONE shared catalog. Top level so joblib can ship it to workers."""
    d  = ds_fn(seed)
    X  = d[feats].to_numpy()
    y  = d[tgt].to_numpy()
    pt = d["p_true"].to_numpy()
    out = {}
    for name, spec in specs.items():
        build = build_from_spec(spec)
        reps = [_metrics_from_oof(y, _oof_predict(build, X, y, seed, r, f"{tag}/{name}"), pt)
                for r in range(SELECT_REPEATS)]
        m = pd.DataFrame(reps).mean().to_dict()
        m["seed"], m["n"] = seed, len(y)
        out[name] = m
    return out

def sweep_plane_strength(plane, strength, seeds=None):
    """Per-seed rows for every config at one (plane, strength). No aggregation here."""
    seeds = list(seeds) if seeds is not None else SELECT_SEEDS
    feats, tgt = PLANE_COLS[plane]
    tasks = [(sc, s) for sc in SCENARIOS for s in seeds]
    t0 = time.time()
    res = Parallel(n_jobs=N_JOBS, prefer="processes")(
        delayed(_eval_configs_one_seed)(
            CONFIGS, dataset_fn(plane, sc, strength), feats, tgt, s,
            f"{plane}/s={strength}/{sc}")
        for sc, s in tasks
    )
    rows = [{"plane": plane, "strength": strength, "config": name, "scenario": sc, **m}
            for (sc, _), per_cfg in zip(tasks, res) for name, m in per_cfg.items()]
    print(f"  {plane:5s} strength={strength}: {len(CONFIGS)} configs x {len(tasks)} "
          f"shared catalogs in {time.time() - t0:.0f}s", flush=True)
    return pd.DataFrame(rows)

# ---- aggregation ----
def scenario_mean_by_seed(ps, metric, configs=None):
    """[seed x config] table of the scenario-mean metric. Pairing across configs preserved."""
    piv = ps.groupby(["seed", "config"])[metric].mean().unstack("config")
    return piv if configs is None else piv[configs]

def family_configs(family):
    return [k for k in CONFIGS if FAMILY_OF[k] == family]

def pick_winner(ps, family, metric=SELECT_METRIC):
    s = scenario_mean_by_seed(ps, metric, family_configs(family)).mean()
    return s.idxmax() if HIGHER_BETTER[metric] else s.idxmin()

def pick_probs(ps, family, metric=SELECT_METRIC, n_boot=4000, boot_seed=0):
    """P(config wins) under resampling of the selection seeds, pairing preserved.
    Five seeds makes this coarse; it is here to stop a 0.004 gap being read as a finding."""
    cols = family_configs(family)
    A = scenario_mean_by_seed(ps, metric, cols).to_numpy()
    rng = np.random.default_rng(boot_seed)
    idx = rng.integers(0, A.shape[0], size=(n_boot, A.shape[0]))
    means = A[idx].mean(axis=1)
    best = means.argmax(1) if HIGHER_BETTER[metric] else means.argmin(1)
    return pd.Series(np.bincount(best, minlength=len(cols)) / n_boot, index=cols)

def family_table(ps, family, k=5):
    """Top k configs by the selection metric, with the other primary metric and P(picked)."""
    cols  = family_configs(family)
    other = [m for m in PRIMARY_METRICS if m != SELECT_METRIC][0]
    t = pd.DataFrame({
        SELECT_METRIC: scenario_mean_by_seed(ps, SELECT_METRIC, cols).mean(),
        other:         scenario_mean_by_seed(ps, other, cols).mean(),
        "P(picked)":   pick_probs(ps, family),
    })
    t.index = [c.split("|", 1)[1] for c in t.index]
    return t.sort_values(SELECT_METRIC,
                         ascending=not HIGHER_BETTER[SELECT_METRIC]).head(k).round(4)

def plot_family_heatmap(ps, family, title):
    cols = family_configs(family)
    t = (ps[ps["config"].isin(cols)]
         .groupby(["config", "scenario"])[SELECT_METRIC].mean().unstack("scenario")[SCENARIOS])
    t.index = [c.split("|", 1)[1] for c in t.index]
    t = t.loc[t.mean(axis=1).sort_values(ascending=not HIGHER_BETTER[SELECT_METRIC]).index]
    fig, ax = plt.subplots(figsize=(7.5, 0.45 * len(t) + 1.5))
    im = ax.imshow(t.values, aspect="auto",
                   cmap="viridis" if HIGHER_BETTER[SELECT_METRIC] else "viridis_r")
    ax.set_xticks(range(len(t.columns))); ax.set_xticklabels(t.columns)
    ax.set_yticks(range(len(t.index)));   ax.set_yticklabels(t.index)
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            ax.text(j, i, f"{t.values[i, j]:.3f}", ha="center", va="center",
                    color="white", fontsize=8)
    ax.set_title(title); fig.colorbar(im, ax=ax)
    plt.tight_layout(); plt.show()

def spec_params(spec):
    s = dict(spec); s.pop("kind")
    return ", ".join(f"{k}={v}" for k, v in s.items())

## 5. Run the selection

One pass per (plane, strength) covering every family, then a pick per family. Nothing here is
reported as performance: these are the seeds the winners were chosen on. Part 2 has the honest
numbers on seeds 5-44.

In [ ]:
SELECT_RAW = {}   # (plane, strength) -> per-seed rows for every config
WINNERS    = {}   # (plane, strength, family) -> "family|config"

t0 = time.time()
for plane in PLANES:
    for st in STRENGTHS:
        ps = sweep_plane_strength(plane, st)
        SELECT_RAW[(plane, st)] = ps
        print("#" * 78)
        print(f"# {plane.upper()} plane, strength={st}   "
              f"(seeds {SELECT_SEEDS[0]}-{SELECT_SEEDS[-1]})")
        print("#" * 78)
        for fam in MODEL_ORDER:
            win = pick_winner(ps, fam)
            WINNERS[(plane, st, fam)] = win
            print(f"===== {fam}: top configs by {SELECT_METRIC} =====")
            display(family_table(ps, fam))
            print(f"  SELECTED -> {fam}: {spec_params(CONFIGS[win])}")
            if SHOW_PLOTS:
                plot_family_heatmap(ps, fam, f"{fam} ({plane}, strength={st}): {SELECT_METRIC}")
        print()
print(f"selection total {time.time() - t0:.0f}s")

# keep the raw rows: the tables above are re-derivable without re-running the grid
pd.concat(SELECT_RAW.values(), ignore_index=True).to_csv("selection_per_seed_raw.csv", index=False)

  xray  strength=1.0: 24 configs x 25 shared catalogs in 672s
##############################################################################
# XRAY plane, strength=1.0   (seeds 0-4)
##############################################################################
===== lr_linear: top configs by prec_at_n =====


,prec_at_n,mse_vs_truth,P(picked)
C10.0,0.1244,0.0017,0.6295
C1.0,0.1240,0.0016,0.3610
C0.1,0.1168,0.0015,0.0095


  SELECTED -> lr_linear: features=plain, C=10.0
===== lr_interaction: top configs by prec_at_n =====


,prec_at_n,mse_vs_truth,P(picked)
C10.0,0.1656,0.0015,0.7075
C1.0,0.1648,0.0013,0.2925
C0.1,0.1588,0.0009,0.0000


  SELECTED -> lr_interaction: features=interaction, C=10.0
===== lr_quadratic: top configs by prec_at_n =====


...

  SELECTED -> lr_quadratic: features=quadratic, C=0.1
===== rf: top configs by prec_at_n =====


,prec_at_n,mse_vs_truth,P(picked)
d3_l5_mfsqrt,0.1524,0.0015,0.5342
dNone_l10_mfsqrt,0.1516,0.0039,0.2622
d6_l10_mfsqrt,0.1508,0.0033,0.1005
d3_l10_mfsqrt,0.1504,0.0015,0.1013
d3_l1_mfsqrt,0.1424,0.0015,0.0000


  SELECTED -> rf: n_estimators=100, max_depth=3, min_samples_leaf=5, max_features=sqrt
===== hgb: top configs by prec_at_n =====


,prec_at_n,mse_vs_truth,P(picked)
d2_lr0.1_l10,0.1368,0.0094,0.4920
d2_lr0.1_l20,0.1368,0.0082,0.3312
d3_lr0.1_l20,0.1336,0.0119,0.1602
dNone_lr0.1_l10,0.1268,0.0379,0.0165
d3_lr0.1_l10,0.1244,0.0143,0.0000


  SELECTED -> hgb: max_depth=2, learning_rate=0.1, min_samples_leaf=10, max_iter=200



## 6. What got picked, per strength

This is the answer to "which hyperparameters got picked at each strength". Three views, because
the point estimate alone is not enough at five seeds:

1. the selected settings per (plane, strength, family);
2. whether the pick moved with strength, with P(picked) for the winner attached;
3. what `mse_vs_truth` would have picked instead, since selection ranks on `prec_at_n` only.

A pick that changes across strength is only meaningful if P(picked) is well clear of its
neighbours. Adjacent configs swap on noise at this sample size.

In [ ]:
sel_rows = []
for (plane, st, fam), win in WINNERS.items():
    ps  = SELECT_RAW[(plane, st)]
    row = ps[ps["config"] == win]
    sel_rows.append({
        "plane": plane, "strength": st, "family": fam,
        "config": win.split("|", 1)[1], "params": spec_params(CONFIGS[win]),
        "select_prec@N": row["prec_at_n"].mean(),
        "select_mse_vs_truth": row["mse_vs_truth"].mean(),
        "select_prec@N_worst_scenario": row.groupby("scenario")["prec_at_n"].mean().min(),
        "P(picked)": pick_probs(ps, fam)[win],
    })
selection_df = (pd.DataFrame(sel_rows)
                .sort_values(["plane", "strength", "family"]).reset_index(drop=True))

print("===== selected settings by strength =====")
display(selection_df.pivot_table(index=["plane", "strength"], columns="family",
                                 values="params", aggfunc="first")[MODEL_ORDER])

print("===== selection-seed metrics for the picked configs "
      "(optimistically biased; Part 2 has the held-out numbers) =====")
display(selection_df.round(4))

print("===== does the pick change across strength? =====")
for plane in PLANES:
    for fam in MODEL_ORDER:
        picks = [WINNERS[(plane, st, fam)] for st in STRENGTHS]
        probs = [pick_probs(SELECT_RAW[(plane, st)], fam)[p] for st, p in zip(STRENGTHS, picks)]
        verdict = "STABLE " if len(set(picks)) == 1 else "CHANGES"
        detail = "  ".join(f"s{st:g}={p.split('|', 1)[1]} (P={pr:.2f})"
                           for st, p, pr in zip(STRENGTHS, picks, probs))
        print(f"  {plane:5s} {fam:15s} {verdict}  {detail}")

print("\n===== would mse_vs_truth have picked the same config? =====")
alt_rows = []
for plane in PLANES:
    for st in STRENGTHS:
        for fam in MODEL_ORDER:
            a = WINNERS[(plane, st, fam)]
            b = pick_winner(SELECT_RAW[(plane, st)], fam, "mse_vs_truth")
            alt_rows.append({"plane": plane, "strength": st, "family": fam,
                             "by_prec_at_n": a.split("|", 1)[1],
                             "by_mse_vs_truth": b.split("|", 1)[1],
                             "agree": a == b})
alt_df = pd.DataFrame(alt_rows)
display(alt_df)
print(f"agreement: {alt_df['agree'].mean():.0%} of the {len(alt_df)} (plane, strength, family) "
      "cells. Where these disagree, every calibration number in Part 2 is for a config chosen "
      "on ranking, not on calibration.")

## 7. Export for Part 2

Two routes, same content. Paste the printed block into Part 2's paste cell, or leave the JSON
next to Part 2 and let it load automatically.

In [ ]:
assert not SMOKE, "SMOKE=True: rehearsal results must not be exported"

LINEUPS = {st: {p: {fam: dict(CONFIGS[WINNERS[(p, st, fam)]]) for fam in MODEL_ORDER}
                for p in PLANES}
           for st in STRENGTHS}
SOURCE  = {st: (f"selection notebook, seeds {SELECT_SEEDS[0]}-{SELECT_SEEDS[-1]}, "
                f"strength={st}, by {SELECT_METRIC}, repo {_sha}")
           for st in STRENGTHS}

pathlib.Path("selection_results.json").write_text(json.dumps({
    "repo_sha": _sha, "strengths": STRENGTHS, "planes": PLANES,
    "select_seeds": SELECT_SEEDS, "report_seeds": REPORT_SEEDS,
    "select_metric": SELECT_METRIC, "select_repeats": SELECT_REPEATS,
    "n_splits": N_SPLITS, "n_at": N_AT, "wise_n": WISE_N,
    "lineups": {str(st): LINEUPS[st] for st in STRENGTHS},
    "source":  {str(st): SOURCE[st] for st in STRENGTHS},
}, indent=2))
selection_df.to_csv("selection_seed_metrics.csv", index=False)
alt_df.to_csv("selection_metric_sensitivity.csv", index=False)
print("wrote: selection_results.json, selection_seed_metrics.csv, "
      "selection_metric_sensitivity.csv, selection_per_seed_raw.csv\n")

def _fmt_spec(spec):
    return "{" + ", ".join(f'"{k}": {v!r}' for k, v in spec.items()) + "}"

out = ["LINEUPS = {"]
for st in STRENGTHS:
    out.append(f"    {st}: {{")
    for p in PLANES:
        out.append(f'        "{p}": {{')
        for fam in MODEL_ORDER:
            out.append(f'            "{fam}": {_fmt_spec(LINEUPS[st][p][fam])},')
        out.append("        },")
    out.append("    },")
out.append("}")
out.append("LINEUP_SOURCE = {")
for st in STRENGTHS:
    out.append(f'    {st}: "{SOURCE[st]}",')
out.append("}")

print("=" * 78)
print("PASTE THE BLOCK BELOW INTO PART 2, SECTION 3")
print("=" * 78)
print("\n".join(out))